# Tutorial 10: AI-Assisted Workflow Composition

## What You Will Learn

- Use AI assistants to accelerate workflow development
- Onboard new components with `onboard_component`
- Diagnose run failures with `diagnose_run`
- Generate human-readable plan explanations
- Compose workflows from natural language
- Migrate manifests between providers
- Understand heuristic vs LLM-enhanced modes

## Prerequisites

- Completed Tutorials 1 and 2
- `pip install scalable[ai]`
- (Optional) LLM API key for enhanced mode

In [ ]:
import os
import tempfile
import json

project_dir = tempfile.mkdtemp(prefix="scalable-ai-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Step 1: AI Backend Configuration

All AI assistants work in two modes:
- **Heuristic** (`no_ai=True`): Deterministic rules, no API calls, fast
- **LLM-enhanced** (`no_ai=False`): Richer output, requires API key

In [ ]:
from scalable.common import settings

# Check AI backend configuration
ai_backend = os.environ.get("SCALABLE_AI_BACKEND", "none")
ai_model = os.environ.get("SCALABLE_AI_MODEL", "(not set)")

print("AI Configuration:")
print(f"  Backend: {ai_backend}")
print(f"  Model: {ai_model}")
print(f"\nAvailable backends:")
print(f"  none   → heuristic only (default, no API needed)")
print(f"  openai → OpenAI API (requires OPENAI_API_KEY)")
print(f"  ollama → Local Ollama (requires SCALABLE_AI_ENDPOINT)")

## Step 2: Check AI Module Availability

In [ ]:
try:
    from scalable.ai import (
        onboard_component,
        diagnose_run,
        explain_plan,
        compose_workflow,
        migrate_manifest,
    )
    print("✓ AI module available")
    print("  Functions: onboard_component, diagnose_run, explain_plan,")
    print("             compose_workflow, migrate_manifest")
except ImportError:
    print("✗ AI module not available")
    print("  Install with: pip install scalable[ai]")
    print("  (requires jinja2 and rich)")

## Step 3: Onboarding a New Component

The `onboard_component` assistant analyzes a model directory and generates component configuration.

In [ ]:
# Create a sample model directory to onboard
model_dir = os.path.join(project_dir, "stitches-model")
os.makedirs(model_dir, exist_ok=True)

# Simulate a model with typical files
with open(os.path.join(model_dir, "run_stitches.R"), "w") as f:
    f.write("# Stitches climate downscaling model\n")
    f.write("library(stitches)\nlibrary(dplyr)\n")
    f.write("result <- run_downscaling(input_path, output_path)\n")

with open(os.path.join(model_dir, "DESCRIPTION"), "w") as f:
    f.write("Package: stitches\n")
    f.write("Title: Climate Downscaling\n")
    f.write("Imports: dplyr, tidyr, ggplot2\n")

with open(os.path.join(model_dir, "Dockerfile"), "w") as f:
    f.write("FROM rocker/r-ver:4.3\n")
    f.write("RUN install2.r stitches dplyr\n")

print(f"Created sample model directory: {model_dir}")
print(f"Files: {os.listdir(model_dir)}")

In [ ]:
try:
    result = onboard_component(
        model_dir,
        name="stitches",
        no_ai=True,  # Heuristic mode
    )
    
    print("Onboarding Result:")
    print(f"  Component YAML:\n{result.component_yaml}")
    if hasattr(result, 'task_yaml'):
        print(f"  Task YAML:\n{result.task_yaml}")
    if hasattr(result, 'recommendations'):
        print(f"  Recommendations: {result.recommendations}")
        
except Exception as e:
    print(f"Onboarding result: {e}")
    print("\nExpected output (heuristic mode):")
    print("  components:")
    print("    stitches:")
    print("      image: (from Dockerfile)")
    print("      cpus: 6")
    print("      memory: 50G")
    print("      tags: [climate, downscaling]")

## Step 4: Diagnosing Run Failures

Generate telemetry with some failures, then diagnose.

In [ ]:
import time
from scalable import ScalableSession
from distributed import as_completed

manifest = """\
version: 1
project:
  name: diagnosis-demo
targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none
components:
  worker:
    cpus: 1
    memory: 1G
tasks:
  compute:
    component: worker
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest)


def failing_task(n: int) -> dict:
    if n % 4 == 0:
        raise RuntimeError(f"OOM: task {n} exceeded memory")
    if n % 7 == 0:
        raise ConnectionError(f"Timeout fetching data for task {n}")
    return {"n": n, "result": n * 42}


# Run with some failures
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()

futures = [client.submit(failing_task, i, tag="worker") for i in range(20)]
for future in as_completed(futures):
    try:
        future.result()
    except Exception:
        pass  # Expected failures

session.close()
print("Run with failures completed.")

In [ ]:
# Diagnose the run
from pathlib import Path
from scalable.telemetry.collectors import iter_run_dirs

runs_dir = Path(".scalable/runs")
run_dirs = sorted(iter_run_dirs(runs_dir))
latest_run = run_dirs[-1] if run_dirs else None

if latest_run:
    try:
        result = diagnose_run(
            run_dir=str(latest_run),
            no_ai=True,
        )
        
        print("Diagnosis Result:")
        print(f"  Summary: {result.summary}")
        if hasattr(result, 'findings'):
            for finding in result.findings:
                print(f"  [{finding.severity}] {finding.category}")
                print(f"    Pattern: {finding.pattern}")
                print(f"    Suggestion: {finding.suggestion}")
    except Exception as e:
        print(f"Diagnosis: {e}")
        # Show what's in the failures file
        failures_file = latest_run / "failures.jsonl"
        if failures_file.exists():
            from scalable.telemetry.collectors import read_jsonl
            failures = read_jsonl(failures_file)
            print(f"\nRaw failures ({len(failures)} events):")
            from collections import Counter
            by_class = Counter(f.get("failure_class", "?") for f in failures)
            for cls, cnt in by_class.most_common():
                print(f"  {cls}: {cnt}")

## Step 5: Explaining Execution Plans

In [ ]:
# Generate a plan
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
plan = session.plan(dry_run=True)

# Save plan to file
plan_data = {
    "target_name": plan.target_name,
    "provider_name": plan.provider_name,
    "manifest_lock": plan.manifest_lock,
}

with open("plan.json", "w") as f:
    json.dump(plan_data, f, indent=2)

print("Plan saved to plan.json")

# Explain it
try:
    result = explain_plan("plan.json")
    print(f"\nExplanation:\n{result.explanation}")
except Exception as e:
    print(f"\nExplain result: {e}")
    print("\nExpected output:")
    print('  This plan will execute "diagnosis-demo" on a local Dask cluster')
    print('  with 2 workers (1 CPU, 1G memory each). No containers are used.')
    print('  Workers run as threads within a single process.')

## Step 6: Composing Workflows from Natural Language

In [ ]:
try:
    result = compose_workflow(
        "Run GCAM reference scenario for SSP2, "
        "then run Stitches to downscale daily climate data"
    )
    
    print("Composed Workflow:")
    print(f"  Code:\n{result.workflow_code[:500]}...")
    if hasattr(result, 'manifest_additions'):
        print(f"\n  Manifest additions:\n{result.manifest_additions}")
        
except Exception as e:
    print(f"Compose result: {e}")
    print("\nExpected output (heuristic mode):")
    print("  A Python workflow script with:")
    print("  - @cacheable decorated functions")
    print("  - ScalableSession setup")
    print("  - Sequential task submission (GCAM → Stitches)")
    print("  - Suggested component/task manifest additions")

## Step 7: Migrating Between Providers

In [ ]:
# Create a Slurm manifest to migrate
slurm_manifest = """\
version: 1
project:
  name: climate-pipeline
targets:
  hpc:
    provider: slurm
    queue: batch
    account: GCIMS
    walltime: "04:00:00"
    interface: ib0
components:
  gcam:
    cpus: 8
    memory: 32G
    tags: [climate]
tasks:
  run_gcam:
    component: gcam
    cache: true
"""

with open("slurm-manifest.yaml", "w") as f:
    f.write(slurm_manifest)

try:
    result = migrate_manifest(
        "slurm-manifest.yaml",
        to_provider="kubernetes",
    )
    
    print("Migration Result (Slurm → Kubernetes):")
    if hasattr(result, 'migrated_yaml'):
        print(f"\n{result.migrated_yaml}")
    if hasattr(result, 'changes_summary'):
        print(f"\nChanges: {result.changes_summary}")
    if hasattr(result, 'migration_notes'):
        print(f"\nNotes: {result.migration_notes}")
        
except Exception as e:
    print(f"Migration result: {e}")
    print("\nExpected changes (slurm → kubernetes):")
    print("  - Remove: queue, account, walltime, interface")
    print("  - Add: namespace, image, adaptive")
    print("  - Components need 'image' field")
    print("  - Mounts → PVC or cloud storage")

## Step 8: Heuristic Analysis

In [ ]:
try:
    from scalable.ai.heuristics import detect_language, estimate_resources
    
    # Language detection
    lang = detect_language(model_dir)
    print(f"Detected language: {lang}")
    
    # Resource estimation
    resources = estimate_resources(
        model_name="gcam",
        input_size_mb=2048,
        num_scenarios=50,
    )
    print(f"Estimated resources: {resources}")
    
except ImportError:
    print("Heuristics module requires scalable[ai]")
except Exception as e:
    print(f"Heuristics: {e}")
    print("\nThe heuristic engine:")
    print("  - Detects language from file extensions and imports")
    print("  - Estimates resources from known model profiles")
    print("  - Generates component configs from templates")
    print("  - All deterministic, reproducible, no API needed")

## Step 9: Development Workflow Integration

The AI assistants form a smooth development loop:

In [ ]:
workflow = """\
Development Workflow with AI Assistants:
========================================

1. ONBOARD a new model:
   scalable init-component ./new-model --name new-model --no-ai

2. COMPOSE a workflow incorporating it:
   scalable compose "Run existing pipeline then feed results to new-model"

3. VALIDATE the generated configuration:
   scalable validate ./scalable.yaml

4. PLAN and EXPLAIN (for team review):
   scalable plan ./scalable.yaml --target local --dry-run -o plan.json
   scalable explain plan.json

5. RUN locally:
   scalable run ./scalable.yaml --target local --workflow workflow.py

6. If it fails, DIAGNOSE:
   scalable diagnose --latest --no-ai

7. When ready for production, MIGRATE:
   scalable migrate scalable.yaml --to-provider kubernetes
"""

print(workflow)

## Step 10: Validating AI-Generated Output

Always validate AI-generated configurations before running.

In [ ]:
from scalable import ScalableSession

# Simulate AI-generated manifest
generated_manifest = """\
version: 1
project:
  name: ai-generated
targets:
  local:
    provider: local
    max_workers: 4
    threads_per_worker: 2
    processes: false
    containers: none
components:
  gcam:
    cpus: 8
    memory: 32G
    tags: [climate]
  stitches:
    cpus: 6
    memory: 50G
    tags: [downscaling]
tasks:
  run_gcam:
    component: gcam
    cache: true
  run_stitches:
    component: stitches
    cache: true
"""

with open("scalable.yaml", "w") as f:
    f.write(generated_manifest)

# Validate
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
report = session.validate()

if report.ok:
    print("✓ AI-generated config is valid — ready to run")
else:
    print("✗ Generated config has issues:")
    for issue in report.errors:
        print(f"  [{issue.code}] {issue.path}: {issue.message}")
    print("\n  Fix issues before running.")

## Summary

1. **`onboard_component`** — Analyzes model dirs, generates component YAML
2. **`diagnose_run`** — Analyzes failures, identifies patterns, suggests fixes
3. **`explain_plan`** — Human-readable plan explanations for stakeholders
4. **`compose_workflow`** — Generates workflow code from natural language
5. **`migrate_manifest`** — Adapts manifests between providers
6. **Heuristic mode** — Fast, deterministic, no API needed (CI/CD safe)
7. **LLM mode** — Richer output, requires API key (interactive use)
8. **Always validate** — AI output is advisory; validate before running

## Next Steps

- **Tutorial 1**: Start from scratch if you're new
- **Tutorial 2**: Deep-dive into manifest schema that AI generates
- **Tutorial 8**: Deploy AI-generated Kubernetes configs
- **Tutorial 9**: Combine AI composition with ML optimization

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")